# stage4 第二輪重訓 bi-encoder — Colab GPU

用「原訓練 + 破模板」**合併**訓練集重訓 bi-encoder,在獨立評估集驗證**真泛化**。

**先做**:Runtime → Change runtime type → **GPU**(T4 即可)。

> **v2 修正(第一次重訓踩坑)**:破模板 query 要 **append 進**原訓練資料,不是取代。
> 只餵 generalization 1003 筆 → 過擬合生成模板,獨立評估集 ab_eval 退步(0.26→0.07)。
> 本版改合併 29443+1003=30446 筆。

閉環:合併訓練集 → 重訓 →(蒸 rbt3)→ export onnx → 重編房源向量 → 評估。

> **同源鐵則**:重訓出新 bi-encoder → 房源向量必用它重編、前端 onnx 也換,三者同批。
> **守天花板 + 看獨立評估**:unified/holdout 部分同源於重訓資料,**最關鍵看 ab_eval_queries**
> (非重訓同源)。沒進步不落地(誠實:破模板不保證贏)。

## 0. 環境 — clone main(含第二輪資料)+ 裝套件

In [ ]:
!nvidia-smi -L
%cd /content
![ -d nchu-edge-rental-ai ] || git clone https://github.com/eric20041027/nchu-edge-rental-ai.git
%cd /content/nchu-edge-rental-ai
!git checkout main && git pull
!pip -q install 'transformers<5' tokenizers onnx onnxruntime
import torch, json; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
d = json.load(open('data/processed/generalization_queries.json'))
cross = [r for r in d if r.get('feat') and any(x in r['query'] for x in ['校園宿舍','便利商店','銅板','泡麵','飯店','圖書館','百貨','深山'])]
print('訓練 query:', len(d), '| 跨域類比:', len(cross), '(第二輪破模板)')

## 1. 合併訓練資料(原 + 破模板)→ 重訓

> **重要(第一次重訓踩的坑)**:破模板 query 要 **append 進**原訓練資料,**不是取代**。
> 只餵 generalization 1003 筆 → 模型過擬合生成模板,在獨立評估集(ab_eval_queries)
> 退步(0.26→0.07)。roadmap 階段④意圖也明寫「**補進**訓練資料」。
>
> 合併:recommendation_train 29443 筆(原真實分佈)+ generalization 1003 筆(破模板)
> = 30446 筆 → 兼得原分佈 + 破模板泛化。schema 已驗相容。

In [ ]:
import json
# 合併:原訓練 + 破模板(append,非取代)。共同欄位 query/property/label/is_hard 即 train 所需。
orig = json.load(open('data/processed/recommendation_train.json'))
gen  = json.load(open('data/processed/generalization_queries.json'))
merged = orig + gen
json.dump(merged, open('data/processed/train_merged_r2.json', 'w'), ensure_ascii=False)
print(f'合併訓練集: {len(orig)} 原 + {len(gen)} 破模板 = {len(merged)} 筆 → train_merged_r2.json')

import os
os.environ['TRAIN_DATA_PATH'] = 'data/processed/train_merged_r2.json'
!TRAIN_DATA_PATH=data/processed/train_merged_r2.json \
    python -m pipeline.model_training.train_bi_encoder \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-length 64 --bf16 --tf32 \
    --output-dir saved_models/rbt6_bi_encoder_r2
# 觀察 log:dev recall_at_1_vs_pool;loss 收斂。
# (此為 6 層 rbt6 重訓。要維持前端 38MB 體積,跑 step2 蒸 rbt3。)

## 2. (可選)蒸餾成 rbt3 student

現役前端是 rbt3(38MB)。要維持體積,把 step1 的 rbt6 蒸成 rbt3。
若只想先驗「破模板有沒有用」可跳過,直接用 rbt6 評估(step 3 改 --saved-dir 指 r2)。

In [ ]:
!TRAIN_DATA_PATH=data/processed/train_merged_r2.json \
    python -m pipeline.model_training.distill_bi_encoder \
    --teacher-dir saved_models/rbt6_bi_encoder_r2 \
    --output-dir saved_models/rbt3_bi_encoder_r2 \
    --epochs 3 --batch-size 32 --lr 3e-5 --alpha 0.5 --max-length 64 --bf16 --tf32

## 3. Export onnx + 重編房源向量(同源)

STUDENT 改成你要落地的那顆(rbt3_bi_encoder_r2 或 rbt6_bi_encoder_r2)。

In [ ]:
STUDENT = 'saved_models/rbt3_bi_encoder_r2'  # ← 若跳過 step2 改 rbt6_bi_encoder_r2
!python -m pipeline.model_training.export_bi_encoder --saved-dir {STUDENT}
!python -m pipeline.data_prep.build_property_embeddings --saved-dir {STUDENT}
!python -m pipeline.data_prep.build_property_embeddings --check

## 4. 驗真泛化 — holdout + unified(守天花板)

holdout = 生成時隔離、風格刻意不同、絕不進訓練 → 真泛化試金石。
unified = 974 房源客觀 GT(批次後 precision/recall 分桶)。

In [ ]:
print('=== holdout(真泛化) ===')
!python pipeline/evaluation/eval_generalization.py --eval-set tests/fixtures/generalization_holdout.json
print('=== unified(974 客觀) ===')
!python pipeline/evaluation/eval_generalization.py --unified
print('=== A/B vs rule-based(recall gate) ===')
!python pipeline/evaluation/eval_vector_vs_rulebased.py

## 5. Gate + 打包下載

| 指標 | 門檻 | 為何 |
|---|---|---|
| **ab_eval_queries all Recall@30** | **≥ 現役 0.26**(別退步) | ⭐ 最獨立評估 — 非重訓資料同源,真泛化試金石 |
| unified 總分 | ≥ 重訓前 0.196 | 客觀 GT(但部分同源於重訓) |
| holdout NDCG@5 | 高 | 風格隔離但維度同源,參考 |
| eval_vector_vs_rulebased | 仍 GO | |
| onnx 大小 | rbt3 ~38MB / rbt6 ~57MB | |

> **⭐ 最關鍵看 ab_eval_queries(all Recall@30)** — 第一次重訓在 unified/holdout 大進步
> 但 ab_eval 退步(0.26→0.07),因只餵 generalization=過擬合生成模板。**合併訓練後
> ab_eval 必須 ≥ 現役 0.26 才算真進步**,否則仍是假泛化,不落地。

**沒進步就不落地** — 誠實面對「破模板不保證贏」(stage4 0.739 天花板)。
全過才下載三檔(onnx + 重編向量),版本同批。

In [ ]:
import os, json
sz = os.path.getsize('frontend/models/bi_encoder_dir/bi_encoder_quant.onnx')/1e6
emb = json.load(open('frontend/assets/property_embeddings.json'))
pd = len(json.load(open('frontend/assets/property_data.json')))
print(f'onnx={sz:.1f}MB | embeddings={emb["count"]} vs property_data={pd}')
assert emb['count'] == pd, '向量數 != 房源數!同源錯配'
!cd /content/nchu-edge-rental-ai && tar czf /content/stage4_r2_artifacts.tgz \
    frontend/models/bi_encoder_dir/bi_encoder_quant.onnx \
    frontend/assets/property_embeddings.json
from google.colab import files; files.download('/content/stage4_r2_artifacts.tgz')

## 落地(本機,gate 全過才做)

```bash
cd <repo>
git checkout -b claude/stage4-r2-retrain-land
tar xzf ~/Downloads/stage4_r2_artifacts.tgz   # 覆蓋 onnx + 房源向量
git add frontend/models/bi_encoder_dir/bi_encoder_quant.onnx frontend/assets/property_embeddings.json
git commit -m 'feat(stage4-r2): 破模板重訓 bi-encoder 落地(holdout 真泛化↑)'
```

把 step4 的 holdout / unified / GO 數字貼回對話,一起對 gate 驗收再開 PR。
⚠️ onnx 與 property_embeddings 必須同批(同一次重訓產出),否則召回壞。